# HH.ru Job Market Analysis
NLP-анализ вакансий: EDA, TF-IDF, семантический матчинг резюме и вакансий.

**Датасет:** [hh.ru vacancies SQLite](https://github.com/sunshineinabagg/hh_analytics) — 392k вакансий

**Стек:** pandas, sklearn, sentence-transformers, shap

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
from collections import Counter
warnings.filterwarnings("ignore")

DB_PATH = "db.sqlite"  # путь к базе данных
conn = sqlite3.connect(DB_PATH)
df = pd.read_sql("SELECT * FROM vacancies", conn)

for col in df.columns:
    if df[col].dtype == object:
        df[col] = df[col].str.strip('"').replace("None", np.nan)

df["salary_bottom"] = pd.to_numeric(df["salary_bottom"], errors="coerce")
df["salary_top"]    = pd.to_numeric(df["salary_top"],    errors="coerce")
df["salary_mid"]    = df[["salary_bottom", "salary_top"]].mean(axis=1)
df["published_at"]  = pd.to_datetime(df["published_at"], errors="coerce")

print(f"Строк: {len(df)} | С зарплатой: {df['salary_mid'].notna().sum()}")
print(df.dtypes)

In [ ]:
top_cities = df["city"].value_counts().head(15)
plt.figure(figsize=(10, 5))
top_cities.plot(kind="bar", color="steelblue")
plt.title("Топ-15 городов")
plt.xlabel("Город"); plt.ylabel("Вакансий")
plt.xticks(rotation=45, ha="right")
plt.tight_layout(); plt.show()

In [ ]:
salary_data = df[
    df["salary_mid"].notna() &
    (df["currency"] == "RUR") &
    (df["salary_mid"] < 500_000)
]
plt.figure(figsize=(10, 5))
plt.hist(salary_data["salary_mid"], bins=50, color="steelblue", edgecolor="white")
plt.title("Распределение зарплат (RUB)")
plt.xlabel("Зарплата"); plt.ylabel("Вакансий")
plt.tight_layout(); plt.show()
print(salary_data["salary_mid"].describe())

In [ ]:
all_skills = []
for skills in df["key_skills"].dropna():
    all_skills.extend([s.strip() for s in skills.split(",") if s.strip()])

top_skills = pd.DataFrame(Counter(all_skills).most_common(20), columns=["skill", "count"])
plt.figure(figsize=(10, 6))
plt.barh(top_skills["skill"][::-1], top_skills["count"][::-1], color="steelblue")
plt.title("Топ-20 навыков")
plt.xlabel("Упоминаний")
plt.tight_layout(); plt.show()

## ML: предсказание уровня опыта по навыкам

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

ml_df = df[df["key_skills"].notna() & df["experience"].notna()].copy()
print(ml_df["experience"].value_counts())

In [ ]:
X_text = ml_df["key_skills"].fillna("")
y = ml_df["experience"]

tfidf = TfidfVectorizer(max_features=1000, ngram_range=(1, 2))
X_tfidf = tfidf.fit_transform(X_text)
print(f"Matrix: {X_tfidf.shape}")

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(LogisticRegression(max_iter=1000, random_state=42),
                         X_tfidf, y, cv=cv, scoring="f1_macro")
print(f"F1-macro (skills only): {scores.mean():.3f} ± {scores.std():.3f}")

In [ ]:
clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_tfidf, y)
feature_names = tfidf.get_feature_names_out()
classes = clf.classes_

fig, axes = plt.subplots(1, len(classes), figsize=(16, 6))
for i, (cls, ax) in enumerate(zip(classes, axes)):
    top_idx = clf.coef_[i].argsort()[-10:]
    ax.barh(feature_names[top_idx], clf.coef_[i][top_idx], color="steelblue")
    ax.set_title(f"Опыт: {cls}")
    ax.set_xlabel("Коэффициент")
plt.suptitle("Навыки по уровню опыта", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
ml_df["features"] = (
    ml_df["name"].fillna("") + " " +
    ml_df["key_skills"].fillna("") + " " +
    ml_df["professional_role"].fillna("")
)
tfidf2 = TfidfVectorizer(max_features=3000, ngram_range=(1, 2))
X_tfidf2 = tfidf2.fit_transform(ml_df["features"])

scores2 = cross_val_score(LogisticRegression(max_iter=1000, random_state=42),
                          X_tfidf2, y, cv=cv, scoring="f1_macro")
print(f"Skills only:          F1 = {scores.mean():.3f}")
print(f"Skills + name + role: F1 = {scores2.mean():.3f}")

In [ ]:
ml_df["level"] = ml_df["experience"].map({
    "noExperience": "Junior", "between1And3": "Junior",
    "between3And6": "Senior", "moreThan6":    "Senior"
})
y_bin = ml_df["level"]

scores_bin = cross_val_score(
    LogisticRegression(max_iter=1000, random_state=42),
    X_tfidf2, y_bin, cv=cv, scoring="roc_auc"
)
print(f"Junior vs Senior — CV ROC-AUC: {scores_bin.mean():.3f} ± {scores_bin.std():.3f}")

## SHAP: интерпретируемость модели

In [ ]:
import shap
clf_final = LogisticRegression(max_iter=1000, random_state=42)
clf_final.fit(X_tfidf2, y_bin)

explainer = shap.LinearExplainer(clf_final, X_tfidf2)
shap_values = explainer.shap_values(X_tfidf2[:500])
shap.summary_plot(shap_values, X_tfidf2[:500].toarray(),
                  feature_names=tfidf2.get_feature_names_out(), max_display=15)

In [ ]:
results = {
    "TF-IDF (skills only)": 0.405,
    "TF-IDF (skills + name + role)": 0.531,
    "Junior vs Senior (AUC)": 0.837,
}
fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(list(results.keys()), list(results.values()),
               color=["#E24B4A", "#F5A623", "#1D9E75"])
ax.axvline(0.5, color="gray", linestyle="--", lw=1)
ax.set_xlabel("Score"); ax.set_title("Model comparison")
ax.set_xlim(0, 1)
for bar, val in zip(bars, results.values()):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f"{val:.3f}", va="center")
plt.tight_layout(); plt.show()

## Semantic Resume–Vacancy Matching

In [ ]:
# !pip install sentence-transformers shap

In [ ]:
HR_STOPWORDS = {
    "приглашаем","ищем","требуется","вакансия","предлагаем","оформление",
    "трудоустройство","отпуск","больничный","зарплата","премия","бонус",
    "дружный","коллектив","молодой","динамичный","стабильный","офис",
    "коммуникабельный","ответственный","пунктуальный","стрессоустойчивый",
    "инициативный","внимательный","обучаемый","работа","обязанности",
    "требования","условия","график","дмс","страховка","питание","фитнес",
    "динамично","развивающаяся","коммуникабельны","ответственны",
    "официальное","корпоративное","компания","организация",
    "в","на","с","по","для","от","до","из","за",
    "наш","мы","вы","они","это","свой","и","или","но","а",
}

def preprocess_text(text):
    if not isinstance(text, str): return ""
    text = text.lower()
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"[\+\d]?(\d{2,3}[-\.\s]?){2,}\d{2,4}", " ", text)
    text = re.sub(r"[^\w\s\+\#]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def full_preprocess(text):
    text = preprocess_text(text)
    words = text.split()
    text = " ".join(w for w in words if w not in HR_STOPWORDS)
    return re.sub(r"\s+", " ", text).strip()

print(full_preprocess("Приглашаем в дружный коллектив! Требования: Python, FastAPI."))

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

vacancy_text = """
Junior Python developer. Requirements: Python, SQL, Git, REST API,
FastAPI or Django, PostgreSQL, OOP. 1+ year experience. Moscow.
"""

resume_text = """
1.5 years commercial dev. Stack: Python, FastAPI, PostgreSQL,
SQLAlchemy, Git, Docker. Telegram bots (aiogram), REST APIs. OOP.
"""

def match_score(vacancy, resume, model):
    score = cosine_similarity(
        model.encode([full_preprocess(vacancy)]),
        model.encode([full_preprocess(resume)])
    )[0][0]
    if score > 0.70:   label = "Strong match"
    elif score > 0.55: label = "Good match — refine resume"
    elif score > 0.35: label = "Partial match"
    else:              label = "Weak match"
    return score, label

score, label = match_score(vacancy_text, resume_text, model)
print(f"Score: {score:.2%} — {label}")

In [ ]:
def gap_analysis(vacancy, resume):
    vec = TfidfVectorizer(ngram_range=(1, 2), max_features=200)
    vec.fit([vacancy, resume])
    v = vec.transform([vacancy]).toarray()[0]
    r = vec.transform([resume]).toarray()[0]
    names = vec.get_feature_names_out()

    matches = [(names[i], v[i]) for i in range(len(names)) if v[i] > 0.05 and r[i] > 0]
    gaps    = [(names[i], v[i]) for i in range(len(names)) if v[i] > 0.05 and r[i] == 0]

    print("Matches:", [w for w, _ in sorted(matches, key=lambda x: -x[1])[:8]])
    print("Gaps:   ", [w for w, _ in sorted(gaps,    key=lambda x: -x[1])[:8]])

gap_analysis(vacancy_text, resume_text)

In [ ]:
sample = df[df["key_skills"].notna()].sample(5000, random_state=42).copy()
sample["vacancy_text"] = (
    sample["name"].fillna("") + " " +
    sample["key_skills"].fillna("") + " " +
    sample["professional_role"].fillna("")
)
sample["vacancy_clean"] = sample["vacancy_text"].apply(full_preprocess)

vacancy_embeddings = model.encode(sample["vacancy_clean"].tolist(),
                                   batch_size=64, show_progress_bar=True)

def find_top_vacancies(resume, embeddings, df, model, threshold=0.60, top_n=3):
    scores = cosine_similarity(model.encode([full_preprocess(resume)]), embeddings)[0]
    result = df.copy()
    result["score"] = scores
    result = result.sort_values("score", ascending=False)

    n = (scores >= threshold).sum()
    print(f"Suitable (>{threshold:.0%}): {n} / {len(scores)} ({n/len(scores)*100:.1f}%)
")

    for i, (_, row) in enumerate(result.head(top_n).iterrows(), 1):
        sal = f"| {int(row['salary_mid']):,} RUB" if pd.notna(row.get("salary_mid")) and row.get("salary_mid", 0) > 0 else ""
        print(f"#{i} [{row['score']:.2%}] {row['name']}")
        print(f"    {row['employer_name']} | {row['city']} {sal}")
        print(f"    {str(row['key_skills'])[:80]}...
")
    return result

results = find_top_vacancies(resume_text, vacancy_embeddings, sample, model)